# HaulMark — Top-Tier Upgraded Pipeline (Debugged)

## Improvements over baseline
- **PART 1** — Blended sequential drift correction (no raw-pred chaining)
- **PART 2** — EWM, trend diff, history-depth / cold-start features
- **PART 3** — Workload, interaction, squared & log-transform features
- **PART 4** — Shift-aware interaction features
- **PART 5** — Enhanced telemetry: speed bins, stop stats, route variability, jerk
- **PART 6** — CatBoost (primary) + XGB + LGB ensemble, multi-seed, dual-target blending
- **PART 7** — Time-series CV with rolling cutoffs
- **PART 8** — Outlier-shrinkage prediction stabilisation
- **PART 9** — Analog dump signal: orientation-based dump-event detection (available from Feb 17)
- **[NEW] PART 11** — Per-vehicle target normalisation: model trained on `acons / veh_avg` (relative behaviour); scale restored at inference via `pred_norm × veh_mean`: orientation-based dump-event detection (available from Feb 17)

## Bugs fixed vs previous draft
1. **Speed-bin fractions** — lambda incorrectly referenced outer `df` via closure; now pre-computed as per-row fractions before `agg()`
2. **EWM closure-in-loop** — `span` captured by reference in `transform` lambda; fixed with default-arg capture `lambda x, s=span:`
3. **XGB `xgb.train` params** — sklearn-style keys (`n_jobs`, `n_estimators`) passed to native booster; replaced with correct native param names
4. **`lag_dist_1` falsy check** — `if lf.get('lag_dist_1')` silently failed when distance = 0.0; fixed to `pd.notna(lf.get('lag_dist_1'))`
5. **`xgb.DMatrix` NameError** — called even when `HAS_XGB=False`; guarded with `if xgb_models and HAS_XGB:`
6. **`is_dumping` sentinel** — rows before Feb 17 are marked with `-1` (no sensor); aggregations guard against these using `x >= 0` masks so pre-cutoff shifts contribute 0, not noise.


In [1]:
# ── Imports ─────────────────────────────────────────────────────────
from collections import defaultdict
import numpy as np
import pandas as pd
import glob, gc, warnings, os

from catboost import CatBoostRegressor   # PRIMARY MODEL
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

# Optional heavy models — graceful fallback if not installed
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("LightGBM not found — will skip LGB in ensemble")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not found — will skip XGB in ensemble")

# ── Config ─────────────────────────────────────────────────────────
DATA_PATH = "/kaggle/input/competitions/mindshift-analytics-haul-mark-challenge/"

# [MODIFIED] Blended history update weights (Part 1)
BLEND_PRED   = 0.70
BLEND_LAG1   = 0.20
BLEND_VEHAVG = 0.10

# [MODIFIED] Stability threshold
STABILITY_THRESHOLD = 3.5
STABILITY_ALPHA     = 0.90

# [NEW] Ensemble weights — CatBoost is now primary
ENS_W_CAT = 0.50
ENS_W_XGB = 0.30
ENS_W_LGB = 0.20

# [NEW] Multi-seed list
SEEDS = [42, 123, 2024]

print("Config loaded.")
print("All files in competition folder:")
for f in sorted(os.listdir(DATA_PATH)):
    print(f"  {f}")

Config loaded.
All files in competition folder:
  fleet.csv
  id_mapping.csv
  id_mapping_new.csv
  mine_001_anonymized.gpkg
  mine_002_anonymized.gpkg
  rfid_refuels_2026-01-01_2026-02-28.parquet
  rfid_refuels_2026-01-01_2026-03-31.parquet
  sample_telemetry.csv
  smry_feb_train_ordered.csv
  smry_jan_train_ordered.csv
  smry_mar_train_ordered.csv
  telemetry_2026-01-01_2026-01-10.parquet
  telemetry_2026-01-11_2026-01-20.parquet
  telemetry_2026-01-21_2026-01-31.parquet
  telemetry_2026-02-01_2026-02-10.parquet
  telemetry_2026-02-11_2026-02-20.parquet
  telemetry_2026-02-21_2026-02-28.parquet
  telemetry_2026-03-01_2026-03-11.parquet
  telemetry_2026-03-12_2026-03-20.parquet


## Block 1 — Load Summary Files (Ground Truth)

In [2]:
jan = pd.read_csv(DATA_PATH + "smry_jan_train_ordered.csv")
feb = pd.read_csv(DATA_PATH + "smry_feb_train_ordered.csv")
mar = pd.read_csv(DATA_PATH + "smry_mar_train_ordered.csv")

print("Jan shape:", jan.shape, "| Columns:", jan.columns.tolist())
print("Feb shape:", feb.shape)
print("Mar shape:", mar.shape)
print("\nSample Jan rows:")
jan.head(4)

Jan shape: (1797, 10) | Columns: ['vehicle', 'mine', 'date', 'shift', 'initlev', 'endlev', 'arefill', 'runhrs', 'acons', 'lph']
Feb shape: (1797, 10)
Mar shape: (1050, 10)

Sample Jan rows:


,vehicle,mine,date,shift,initlev,endlev,arefill,runhrs,acons,lph
0,Dump002,mine001,2026-01-01,C,443.24,450.00,0.0,0.0,0.00,NaN
1,Dump002,mine001,2026-01-01,A,450.00,643.24,200.0,0.0,6.76,inf
2,Dump002,mine001,2026-01-01,B,643.24,645.95,0.0,0.0,0.00,NaN
3,Dump002,mine001,2026-01-02,C,645.95,648.65,0.0,0.0,0.00,NaN


In [3]:
smry = pd.concat([jan, feb, mar], ignore_index=True)
del jan, feb, mar; gc.collect()

print("All columns in summary:", smry.columns.tolist())
print("\nShape:", smry.shape)
smry.head(4)

All columns in summary: ['vehicle', 'mine', 'date', 'shift', 'initlev', 'endlev', 'arefill', 'runhrs', 'acons', 'lph']

Shape: (4644, 10)


,vehicle,mine,date,shift,initlev,endlev,arefill,runhrs,acons,lph
0,Dump002,mine001,2026-01-01,C,443.24,450.00,0.0,0.0,0.00,NaN
1,Dump002,mine001,2026-01-01,A,450.00,643.24,200.0,0.0,6.76,inf
2,Dump002,mine001,2026-01-01,B,643.24,645.95,0.0,0.0,0.00,NaN
3,Dump002,mine001,2026-01-02,C,645.95,648.65,0.0,0.0,0.00,NaN


In [4]:
print("acons stats:")
print(smry['acons'].describe().round(1))
print("\nNull acons rows:", smry['acons'].isnull().sum())
print("\nUnique shifts:", smry['shift_dpr'].unique() if 'shift_dpr' in smry.columns
      else smry['shift'].unique() if 'shift' in smry.columns else "SHIFT COL NOT FOUND")
print("\nDate range:", smry['date_dpr'].min() if 'date_dpr' in smry.columns
      else smry['date'].min(), "→",
      smry['date_dpr'].max() if 'date_dpr' in smry.columns else smry['date'].max())

acons stats:
count    4644.0
mean      141.6
std       109.9
min         0.0
25%         4.4
50%       156.0
75%       236.7
max       602.6
Name: acons, dtype: float64

Null acons rows: 0

Unique shifts: ['C' 'A' 'B']

Date range: 2026-01-01 → 2026-03-11


In [5]:
if 'date_dpr' in smry.columns:  smry = smry.rename(columns={'date_dpr': 'date'})
if 'shift_dpr' in smry.columns: smry = smry.rename(columns={'shift_dpr': 'shift'})

smry['date'] = pd.to_datetime(smry['date'])
smry = smry[smry['acons'].notna() & (smry['acons'] > 0)].copy()
smry = smry[smry['vehicle'].str.startswith('Dump')].copy()

print(f"Summary rows after cleaning : {len(smry)}")
print(f"Unique vehicles             : {smry['vehicle'].nunique()}")
print(f"Date range                  : {smry['date'].min().date()} → {smry['date'].max().date()}")
print(f"acons: mean={smry['acons'].mean():.1f}  std={smry['acons'].std():.1f}  "
      f"min={smry['acons'].min():.1f}  max={smry['acons'].max():.1f}")
smry[['vehicle','date','shift','acons']].head(6)

Summary rows after cleaning : 3819
Unique vehicles             : 32
Date range                  : 2026-01-01 → 2026-03-11
acons: mean=172.2  std=97.1  min=0.0  max=602.6


,vehicle,date,shift,acons
1,Dump002,2026-01-01,A,6.76
4,Dump002,2026-01-02,A,5.41
10,Dump002,2026-01-04,A,126.22
11,Dump002,2026-01-04,B,74.36
13,Dump002,2026-01-05,A,166.65
14,Dump002,2026-01-05,B,186.85


## Block 2 — Load Fleet Metadata

In [6]:
fleet = pd.read_csv(DATA_PATH + "fleet.csv")
print("Fleet shape:", fleet.shape)
print("Columns:", fleet.columns.tolist())
print(fleet.head(10))

Fleet shape: (75, 5)
Columns: ['fleet', 'vehicle', 'tankcap', 'dump_switch', 'mine_anon']
            fleet  vehicle  tankcap  dump_switch mine_anon
0  Backhoe Loader   BHL003      NaN          NaN   mine002
1  Backhoe Loader   BHL001      NaN          NaN   mine001
2  Backhoe Loader   BHL005      NaN          NaN   mine002
3  Backhoe Loader   BHL004      NaN          NaN   mine002
4  Backhoe Loader   BHL006      NaN          NaN   mine002
5  Backhoe Loader   BHL002      NaN          NaN   mine001
6          Dumper  Dump001   1311.0          1.0   mine001
7          Dumper  Dump002   1310.0          1.0   mine001
8          Dumper  Dump003   1311.0          1.0   mine001
9          Dumper  Dump004   1311.0          1.0   mine001


In [7]:
fleet = fleet[fleet['vehicle'].str.startswith('Dump')].copy()
print(f"Dump vehicles in fleet: {len(fleet)}")
print(fleet)

Dump vehicles in fleet: 43
     fleet  vehicle  tankcap  dump_switch mine_anon
6   Dumper  Dump001   1311.0          1.0   mine001
7   Dumper  Dump002   1310.0          1.0   mine001
8   Dumper  Dump003   1311.0          1.0   mine001
9   Dumper  Dump004   1311.0          1.0   mine001
10  Dumper  Dump005   1311.0          1.0   mine001
11  Dumper  Dump006   1311.0          1.0   mine001
12  Dumper  Dump007   1311.0          1.0   mine001
13  Dumper  Dump008   1311.0          1.0   mine001
14  Dumper  Dump009   1311.0          1.0   mine001
15  Dumper  Dump010   1311.0          1.0   mine001
16  Dumper  Dump029   1311.0          NaN   mine002
17  Dumper  Dump011   1311.0          1.0   mine001
18  Dumper  Dump012   1310.0          1.0   mine001
19  Dumper  Dump030   1311.0          NaN   mine002
20  Dumper  Dump014   1311.0          1.0   mine001
21  Dumper  Dump015   1311.0          1.0   mine001
22  Dumper  Dump031   1311.0          NaN   mine002
23  Dumper  Dump016   1310.0         

## Block 3 — [IMPROVED + ADVANCED] Telemetry Feature Extraction

**New in this version (Part 5):**
- Speed bins (% time in low/med/high speed bands)
- Stop duration statistics (avg, max)
- Route variability (std lat/lon)
- Jerk features (change in acceleration magnitude)

**Bug fix:** Speed-bin fractions are now pre-computed as per-ping fraction columns
before aggregation, eliminating the broken closure over the outer `df` variable.

In [8]:
def process_one_file(filepath):
    """
    [IMPROVED] Load one telemetry parquet file.
    Compute physics + Part-5 features per (vehicle, date, shift).
    Memory-safe: raw df is deleted before returning.
    """
    df = pd.read_parquet(filepath)
    df = df[df['vehicle'].str.startswith('Dump')].copy()
    if df.empty:
        return pd.DataFrame()

    # Downcast to float32 — ~40% RAM saving
    for col in ['speed', 'altitude', 'latitude', 'longitude',
                'disthav', 'axis_x', 'axis_y', 'axis_z']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

    # Detect timestamp column robustly
    ts_col = next(
        (c for c in df.columns if c.lower() in
         ('ts', 'timestamp', 'datetime', 'event_time', 'time', 'recorded_at')),
        None
    )
    if ts_col is None:
        return pd.DataFrame()

    df['ts'] = pd.to_datetime(df[ts_col])

    # ── Shift + day assignment ──────────────────────────────────────────
    # Shift boundaries (24-h clock):
    #   C : 22:00 – 05:59  (spans midnight; the 22:00-23:59 part belongs to NEXT calendar day)
    #   A : 06:00 – 13:59
    #   B : 14:00 – 21:59
    #
    # Day definition: a "day" runs 22:00 (prev calendar day) → 21:59 (current calendar day).
    # So any ping with hour >= 22 is assigned to the *next* calendar day.
    hour = df['ts'].dt.hour

    def assign_shift_series(h):
        """Assign shift label from an integer hour-of-day Series."""
        shift = pd.Series('', index=h.index, dtype=object)
        shift[(h >= 22) | (h < 6)]   = 'C'   # 22:00-05:59 → C shift
        shift[(h >= 6)  & (h < 14)]  = 'A'   # 06:00-13:59 → A shift
        shift[(h >= 14) & (h < 22)]  = 'B'   # 14:00-21:59 → B shift
        return shift

    if 'shift_dpr' in df.columns:
        df['shift'] = df['shift_dpr'].astype(str)
        mask_bad = df['shift'].isin(['nan', 'None', '', 'Unknown'])
        if mask_bad.any():
            df.loc[mask_bad, 'shift'] = assign_shift_series(hour[mask_bad]).values
    else:
        df['shift'] = assign_shift_series(hour).values

    # Roll date forward by 1 day for pings timestamped 22:00-23:59
    # (they belong to the next day under the "day starts at 22:00" convention)
    base_date = pd.to_datetime(df['ts'].dt.date)
    df['date'] = base_date + pd.to_timedelta((hour >= 22).astype(int), unit='D')

    df = df[df['shift'].isin(['A', 'B', 'C'])].copy()
    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(['vehicle', 'ts'])

    # ── Altitude smoothing ─────────────────────────────────────
    if 'altitude' in df.columns:
        df['alt_smooth'] = (
            df.groupby('vehicle')['altitude']
            .transform(lambda x: x.rolling(3, min_periods=1, center=True).mean())
            .astype('float32')
        )
        df['alt_diff']      = df.groupby('vehicle')['alt_smooth'].diff().fillna(0).astype('float32')
        df['positive_lift'] = df['alt_diff'].clip(lower=0)
        df['negative_lift'] = df['alt_diff'].clip(upper=0).abs()
        del df['alt_diff']
    else:
        df['alt_smooth']    = np.float32(0)
        df['positive_lift'] = np.float32(0)
        df['negative_lift'] = np.float32(0)

    # ── Distance + gradient ─────────────────────────────────────
    if 'disthav' in df.columns:
        df['dist_m'] = df['disthav'].fillna(0).astype('float32')
        df.loc[df['dist_m'] > 500, 'dist_m'] = 0
    else:
        df['dist_m'] = np.float32(0)

    df['gradient'] = (
        df['positive_lift'] / (df['dist_m'] + 1e-3)
    ).clip(-0.3, 0.3).astype('float32')

    # ── Time delta + operating modes ────────────────────────────
    df['prev_ts']      = df.groupby('vehicle')['ts'].shift(1)
    df['time_delta_s'] = (
        (df['ts'] - df['prev_ts']).dt.total_seconds()
        .fillna(0).clip(0, 1800).astype('float32')
    )
    del df['prev_ts']

    if 'speed' in df.columns:
        df['moving_seconds'] = (df['time_delta_s'] * (df['speed'] > 2)).astype('float32')
    else:
        df['moving_seconds'] = np.float32(0)

    if 'ignition' in df.columns and 'speed' in df.columns:
        df['idle_seconds'] = (
            df['time_delta_s'] * ((df['speed'] <= 2) & (df['ignition'] == 1))
        ).astype('float32')
    elif 'speed' in df.columns:
        df['idle_seconds'] = (
            df['time_delta_s'] * (df['speed'] <= 2) * (df['time_delta_s'] <= 300)
        ).astype('float32')
    else:
        df['idle_seconds'] = np.float32(0)

    # Stop-go events
    if 'speed' in df.columns:
        speed_lag           = df.groupby('vehicle')['speed'].shift(1).fillna(0)
        df['stop_go_event'] = ((df['speed'] > 5) & (speed_lag <= 2)).astype('int8')
        del speed_lag
    else:
        df['stop_go_event'] = np.int8(0)

    # Accelerometer roughness + jerk
    if all(c in df.columns for c in ['axis_x', 'axis_y', 'axis_z']):
        df['accel_mag'] = np.sqrt(
            df['axis_x']**2 + df['axis_y']**2 + df['axis_z']**2
        ).astype('float32')
    else:
        df['accel_mag'] = np.float32(0)

    # [NEW PART 5] Jerk = |change in accel magnitude|
    df['accel_prev'] = df.groupby('vehicle')['accel_mag'].shift(1).fillna(0)
    df['jerk']       = (df['accel_mag'] - df['accel_prev']).abs().astype('float32')
    del df['accel_prev']

    # ── [NEW PART 5] Speed bins ────────────────────────────────────
    # FIX: pre-compute per-ping fraction columns to avoid closure over outer df.
    # Previously lambdas did df.loc[x.index, 'time_delta_s'] which is unsafe.
    if 'speed' in df.columns:
        total_td = df['time_delta_s'] + 1.0   # +1 avoids div-by-zero
        df['spd_low_frac_ping']  = (df['time_delta_s'] * (df['speed'] <  20)) / total_td
        df['spd_med_frac_ping']  = (df['time_delta_s'] * ((df['speed'] >= 20) & (df['speed'] < 50))) / total_td
        df['spd_high_frac_ping'] = (df['time_delta_s'] * (df['speed'] >= 50)) / total_td
    else:
        df['spd_low_frac_ping']  = np.float32(0)
        df['spd_med_frac_ping']  = np.float32(0)
        df['spd_high_frac_ping'] = np.float32(0)

    # ── [NEW PART 5] Stop duration ─────────────────────────────────
    if 'speed' in df.columns:
        df['is_stopped']      = (df['speed'] <= 2).astype('int8')
        df['stop_duration_s'] = np.where(
            df['is_stopped'] == 1, df['time_delta_s'], 0.0
        ).astype('float32')
    else:
        df['is_stopped']      = np.int8(0)
        df['stop_duration_s'] = np.float32(0)


    # ── [NEW PART 9] Analog dump signal ────────────────────────────────
    # Orientation-based dump detection via axis_z tilt.
    # Signal only available from 2026-02-17; pre-cutoff rows stay NaN.
    ANALOG_CUTOFF = pd.Timestamp('2026-02-17')
    if 'axis_z' in df.columns:
        # Per-vehicle threshold (avoids cross-vehicle bias)
        # FIX: use transform so threshold is vehicle-specific, not global
        veh_az_mean = df.groupby('vehicle')['axis_z'].transform('mean')
        veh_az_std  = df.groupby('vehicle')['axis_z'].transform('std').fillna(1e-3).clip(lower=1e-3)
        dump_threshold = veh_az_mean + 0.5 * veh_az_std
        df['is_dumping'] = (df['axis_z'] > dump_threshold).astype('int8')
        # Zero-out signal for rows before the cutoff (sensor didn't exist)
        df.loc[df['date'] < ANALOG_CUTOFF, 'is_dumping'] = np.int8(-1)  # -1 = no sensor
        # Pre-compute valid tilt column: NaN where no sensor, axis_z value otherwise
        # FIX: avoids broken closure over outer df inside agg lambdas
        df['dump_tilt_valid'] = np.where(
            df['is_dumping'] >= 0, df['axis_z'].astype('float32'), np.nan
        ).astype('float32')
    else:
        df['is_dumping']      = np.int8(-1)
        df['dump_tilt_valid'] = np.float32(np.nan)

    # ── Aggregate to (vehicle, date, shift) ────────────────────────
    grp = ['vehicle', 'date', 'shift']

    agg = df.groupby(grp).agg(
        total_dist_km   = ('dist_m',             lambda x: x.sum() / 1000),
        positive_lift_m = ('positive_lift',      'sum'),
        negative_lift_m = ('negative_lift',      'sum'),
        alt_range       = ('alt_smooth',         lambda x: x.max() - x.min()),
        alt_std         = ('alt_smooth',         'std'),
        grad_mean       = ('gradient',           'mean'),
        grad_abs_mean   = ('gradient',           lambda x: x.abs().mean()),
        grad_std        = ('gradient',           'std'),
        grad_max        = ('gradient',           'max'),
        speed_mean      = ('speed',              'mean')              if 'speed' in df.columns else ('dist_m', 'mean'),
        speed_max       = ('speed',              'max')               if 'speed' in df.columns else ('dist_m', 'max'),
        speed_std       = ('speed',              'std')               if 'speed' in df.columns else ('dist_m', 'std'),
        speed_p75       = ('speed',              lambda x: x.quantile(0.75)) if 'speed' in df.columns else ('dist_m', lambda x: x.quantile(0.75)),
        moving_hours    = ('moving_seconds',     lambda x: x.sum() / 3600),
        idle_hours      = ('idle_seconds',       lambda x: x.sum() / 3600),
        stop_go_events  = ('stop_go_event',      'sum'),
        accel_roughness = ('accel_mag',          'std'),
        ping_count      = ('ts',                 'count'),
        spd_low_frac    = ('spd_low_frac_ping',  'mean'),
        spd_med_frac    = ('spd_med_frac_ping',  'mean'),
        spd_high_frac   = ('spd_high_frac_ping', 'mean'),
        stop_dur_mean   = ('stop_duration_s',    lambda x: x[x > 0].mean() if (x > 0).any() else 0.0),
        stop_dur_max    = ('stop_duration_s',    'max'),
        lat_std         = ('latitude',           'std') if 'latitude'  in df.columns else ('dist_m', lambda x: np.nan),
        lon_std         = ('longitude',          'std') if 'longitude' in df.columns else ('dist_m', lambda x: np.nan),
        jerk_mean       = ('jerk',               'mean'),
        jerk_max        = ('jerk',               'max'),
        jerk_std        = ('jerk',               'std'),
        # [NEW PART 9] Analog dump signal — no outer-df closure
        dump_count       = ('is_dumping',      lambda x: (x == 1).sum()),
        dump_signal_frac = ('is_dumping',      lambda x: (x == 1).sum() / max((x >= 0).sum(), 1) if (x >= 0).any() else np.nan),
        dump_tilt_mean   = ('dump_tilt_valid', 'mean'),
        dump_tilt_max    = ('dump_tilt_valid', 'max'),
        has_analog_data  = ('is_dumping',      lambda x: int((x >= 0).any())),
    ).reset_index()

    del df; gc.collect()
    return agg


tele_files = sorted(glob.glob(DATA_PATH + "telemetry_*.parquet"))
print(f"Found {len(tele_files)} telemetry files")

all_aggs = []
for f in tele_files:
    print(f"  {f.split('/')[-1]}...", end=' ')
    agg = process_one_file(f)
    all_aggs.append(agg)
    print(f"{len(agg)} shift-rows")

tele_features = pd.concat(all_aggs, ignore_index=True) if any(len(a) > 0 for a in all_aggs) else pd.DataFrame()
del all_aggs; gc.collect()

if not tele_features.empty:
    num_cols = [c for c in tele_features.columns if c not in ['vehicle','date','shift']]
    tele_features = tele_features.groupby(['vehicle','date','shift'])[num_cols].mean().reset_index()

    # Core derived features
    tele_features['stop_go_per_km']     = tele_features['stop_go_events'] / (tele_features['total_dist_km'] + 0.1)
    tele_features['idle_fraction']      = tele_features['idle_hours'] / (tele_features['moving_hours'] + tele_features['idle_hours'] + 0.01)
    tele_features['terrain_difficulty'] = tele_features['positive_lift_m'] / (tele_features['total_dist_km'] + 0.1)
    tele_features['net_lift']           = tele_features['positive_lift_m'] - tele_features['negative_lift_m']
    tele_features['efficiency']         = tele_features['total_dist_km'] / (tele_features['moving_hours'] + 0.01)
    tele_features['effort']             = tele_features['total_dist_km'] * (1 + tele_features['idle_fraction'] + tele_features['terrain_difficulty'])
    tele_features['idle_per_km']        = tele_features['idle_hours'] / (tele_features['total_dist_km'] + 0.1)
    tele_features['lift_per_km']        = tele_features['positive_lift_m'] / (tele_features['total_dist_km'] + 0.1)

    print(f"\nTelemetry shift features : {tele_features.shape}")
    print(f"Memory                   : {tele_features.memory_usage(deep=True).sum()/1e6:.1f} MB")
    tele_features.head(3)
else:
    print("No telemetry features extracted — continuing without them.")

Found 8 telemetry files
  telemetry_2026-01-01_2026-01-10.parquet... 830 shift-rows
  telemetry_2026-01-11_2026-01-20.parquet... 875 shift-rows
  telemetry_2026-01-21_2026-01-31.parquet... 940 shift-rows
  telemetry_2026-02-01_2026-02-10.parquet... 891 shift-rows
  telemetry_2026-02-11_2026-02-20.parquet... 891 shift-rows
  telemetry_2026-02-21_2026-02-28.parquet... 708 shift-rows
  telemetry_2026-03-01_2026-03-11.parquet... 981 shift-rows
  telemetry_2026-03-12_2026-03-20.parquet... 783 shift-rows

Telemetry shift features : (6899, 44)
Memory                   : 2.1 MB


## Block 4 — Merge Summary + Telemetry + Fleet

In [9]:
data = pd.merge(
    smry[['vehicle','date','shift','acons']],
    tele_features,
    on=['vehicle','date','shift'],
    how='left'
)

if len(fleet) > 0:
    data = pd.merge(data, fleet, on='vehicle', how='left')
    print("Fleet columns added:", [c for c in fleet.columns if c != 'vehicle'])

print(f"\nMerged dataset shape : {data.shape}")
print(f"Telemetry match rate : {data['total_dist_km'].notna().mean()*100:.1f}%")
print(f"\nacons stats:")
print(data['acons'].describe().round(1))
data[['vehicle','date','shift','acons','total_dist_km','moving_hours','idle_hours']].head(6)

Fleet columns added: ['fleet', 'tankcap', 'dump_switch', 'mine_anon']

Merged dataset shape : (3819, 49)
Telemetry match rate : 99.0%

acons stats:
count    3819.0
mean      172.2
std        97.1
min         0.0
25%       102.0
50%       186.5
75%       248.4
max       602.6
Name: acons, dtype: float64


,vehicle,date,shift,acons,total_dist_km,moving_hours,idle_hours
0,Dump002,2026-01-01,A,6.76,0.020292,0.000000,0.000000
1,Dump002,2026-01-02,A,5.41,0.013388,0.000000,0.000000
2,Dump002,2026-01-04,A,126.22,31.589050,1.528889,0.946111
3,Dump002,2026-01-04,B,74.36,18.420732,1.000833,1.038333
4,Dump002,2026-01-05,A,166.65,43.710274,1.770833,0.889444
5,Dump002,2026-01-05,B,186.85,52.320923,2.278056,0.863333


## Block 5 — [IMPROVED + NEW] Lag, Rolling & Advanced Time-Series Features

**New:** EWM (Part 2), trend diff (Part 2), history depth + cold-start (Part 2),
workload + interactions (Part 3), squared + log transforms (Part 3), shift interactions (Part 4).

**Bug fix:** EWM lambda now uses default-arg capture `lambda x, s=span:` to freeze
the loop variable at each iteration, preventing all EWMs from using span=10.

In [10]:
print("Building lag, rolling, EWM and advanced features...")

# Shift chronological order within a day (day starts at 22:00):
#   C=0 (22:00-05:59), A=1 (06:00-13:59), B=2 (14:00-21:59)
SHIFT_ORDER = {'C': 0, 'A': 1, 'B': 2}
data['shift_num'] = data['shift'].map(SHIFT_ORDER).fillna(0).astype(int)
data = data.sort_values(['vehicle', 'date', 'shift_num']).reset_index(drop=True)

# ── Core lags ───────────────────────────────────────────────────
data['lag_acons_1'] = data.groupby('vehicle')['acons'].shift(1)
data['lag_acons_2'] = data.groupby('vehicle')['acons'].shift(2)
data['lag_acons_3'] = data.groupby('vehicle')['acons'].shift(3)

# ── Rolling (correct per-vehicle pattern) ─────────────────────────────
data['_lag1_tmp']       = data.groupby('vehicle')['acons'].shift(1)
data['rolling_acons_3'] = data.groupby('vehicle')['_lag1_tmp'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())
data['rolling_acons_7'] = data.groupby('vehicle')['_lag1_tmp'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())
data['rolling_acons_std3'] = data.groupby('vehicle')['_lag1_tmp'].transform(
    lambda x: x.rolling(3, min_periods=2).std())
data.drop(columns=['_lag1_tmp'], inplace=True)

# ── [NEW PART 2] EWM features ─────────────────────────────────────
# FIX: default-arg capture `s=span` freezes span at each loop iteration.
# Without this, all three lambdas close over the same variable and all use span=10.
for span in [3, 5, 10]:
    data[f'ewm_acons_{span}'] = (
        data.groupby('vehicle')['acons']
        .transform(lambda x, s=span: x.shift(1).ewm(span=s, min_periods=1).mean())
    )
    print(f"  ewm_acons_{span} created")

# ── [NEW PART 2] Trend features ───────────────────────────────────
data['acons_trend_3_7']  = data['rolling_acons_3'] - data['rolling_acons_7']
data['acons_trend_sign'] = np.sign(data['acons_trend_3_7'])   # NaN-safe: np.sign(NaN)=NaN

# ── [NEW PART 2] History depth & cold-start ────────────────────────
data['history_length'] = data.groupby('vehicle').cumcount()   # 0-indexed
data['is_cold_start']  = (data['history_length'] < 3).astype(int)
print("  history_length + is_cold_start created")

# ── Lag on telemetry features ───────────────────────────────────
data['lag_dist_1']  = data.groupby('vehicle')['total_dist_km'].shift(1)
data['lag_idle_1']  = data.groupby('vehicle')['idle_hours'].shift(1) if 'idle_hours' in data.columns else np.nan

data['_lag_dist_tmp'] = data.groupby('vehicle')['total_dist_km'].shift(1)
data['rolling_dist_3'] = data.groupby('vehicle')['_lag_dist_tmp'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())
data.drop(columns=['_lag_dist_tmp'], inplace=True)

# ── Vehicle-level historical stats ───────────────────────────────
veh_avg      = data.groupby('vehicle')['acons'].mean().rename('vehicle_avg_acons')
veh_std      = data.groupby('vehicle')['acons'].std().rename('vehicle_std_acons')
veh_avg_dist = (
    data.groupby('vehicle')['total_dist_km'].mean().rename('vehicle_avg_dist')
    if 'total_dist_km' in data.columns
    else pd.Series(dtype=float, name='vehicle_avg_dist')
)

data = data.join(veh_avg,      on='vehicle')
data = data.join(veh_std,      on='vehicle')
data = data.join(veh_avg_dist, on='vehicle')

data['dist_ratio']  = data['total_dist_km'] / (data['vehicle_avg_dist'] + 0.1) if 'total_dist_km' in data.columns else np.nan
data['acons_ratio'] = data['lag_acons_1']   / (data['vehicle_avg_acons'] + 0.1)

# ── Time features ───────────────────────────────────────────────
data['day_of_week']  = data['date'].dt.dayofweek
data['is_weekend']   = (data['day_of_week'] >= 5).astype(int)
data['month']        = data['date'].dt.month
data['day_of_month'] = data['date'].dt.day

# ── [NEW PART 3] Workload & interaction features ─────────────────────
if 'total_dist_km' in data.columns and 'idle_hours' in data.columns and 'positive_lift_m' in data.columns:
    data['workload'] = (
        data['total_dist_km'].fillna(0) +
        data['idle_hours'].fillna(0) * 10 +
        data['positive_lift_m'].fillna(0) / 100
    )
    data['idle_x_dist'] = data['idle_hours'].fillna(0)      * data['total_dist_km'].fillna(0)
    data['lift_x_dist'] = data['positive_lift_m'].fillna(0) * data['total_dist_km'].fillna(0)
    print("  workload, idle_x_dist, lift_x_dist created")

data['fuel_per_km_lag1'] = data['lag_acons_1'] / (data['lag_dist_1'].fillna(0) + 0.1)

# ── [NEW] High-load trigger flag (Part 2) ────────────────────────────
# Marks shifts in the top-20% of distance OR idle hours.
# Gives LightGBM an explicit binary signal for high-consumption regimes.
# Quantile thresholds are locked to the full training distribution.
_dist_q80 = data['total_dist_km'].quantile(0.8) if 'total_dist_km' in data.columns else np.inf
_idle_q80 = data['idle_hours'].quantile(0.8)    if 'idle_hours'    in data.columns else np.inf
data['high_load_flag'] = (
    (data['total_dist_km'].fillna(0) > _dist_q80) |
    (data['idle_hours'].fillna(0)    > _idle_q80)
).astype(int)
print(f"  high_load_flag created  (dist_q80={_dist_q80:.1f} km, idle_q80={_idle_q80:.2f} h, "
      f"flagged={data['high_load_flag'].mean()*100:.1f}%)")

# ── [NEW PART 3] Nonlinear transforms ───────────────────────────────
for col in ['total_dist_km', 'positive_lift_m', 'idle_hours', 'workload']:
    if col in data.columns:
        data[f'{col}_sq']  = data[col].fillna(0) ** 2
        data[f'{col}_log'] = np.log1p(data[col].fillna(0).clip(lower=0))
print("  squared + log features created")

# ── [NEW PART 4] Shift-interaction features ──────────────────────────
# DESIGN: shift interaction features over separate per-shift models.
# On small datasets separate models give 3× less training data and high variance.
# Explicit cross-terms let LGB split more efficiently.
if 'total_dist_km' in data.columns:
    data['shift_x_dist']    = data['shift_num'] * data['total_dist_km'].fillna(0)
    data['shift_x_idle']    = data['shift_num'] * data['idle_hours'].fillna(0)       if 'idle_hours'      in data.columns else 0
    data['shift_x_lift']    = data['shift_num'] * data['positive_lift_m'].fillna(0)  if 'positive_lift_m' in data.columns else 0
    data['shift_x_lag1']    = data['shift_num'] * data['lag_acons_1'].fillna(0)
    data['shift_x_weekend'] = data['shift_num'] * data['is_weekend']
    print("  shift interaction features created")

# ── [NEW] Vehicle × shift mean encoding ────────────────────────────
veh_shift_avg = (
    data.groupby(['vehicle', 'shift'])['acons']
    .mean()
    .rename('veh_shift_avg_acons')
    .reset_index()
)
data = data.merge(veh_shift_avg, on=['vehicle','shift'], how='left')
print("  vehicle×shift mean encoding created")


# ── [NEW PART 9] Analog signal availability flag ───────────────────
ANALOG_SIGNAL_START = pd.Timestamp('2026-02-17')
data['has_analog_signal'] = (data['date'] >= ANALOG_SIGNAL_START).astype(int)

# Fill dump signal features with 0 / NaN for pre-cutoff rows (no sensor data)
dump_signal_cols = ['dump_count', 'dump_signal_frac', 'dump_tilt_mean', 'dump_tilt_max', 'has_analog_data']
for col in dump_signal_cols:
    if col in data.columns:
        # Rows before cutoff: sensor didn't exist — fill with 0 so model learns absence
        # Leave as NaN — LGB handles missing natively; 0 would falsely signal 'no dumps'
        pass  # NaN kept intentionally for pre-cutoff rows
print('  analog signal flag + dump feature nulls handled')

print(f"\nDataset with all features : {data.shape}")
print(f"Cold-start rows           : {data['is_cold_start'].sum()}")

# ── [NEW PART 11] Per-vehicle target normalisation ──────────────────────────
# MUST be computed on the full `data` frame (before train/val split) so
# veh_avg_for_norm is derived from ALL available history, not just train.
# The normalised target is used for training; scale is recovered at inference.
#
# Why this helps:
#   - Raw acons spans ~0–600 L across vehicles with very different baselines
#   - A single model must simultaneously handle a Dump002 mean of ~80 L and a
#     Dump031 mean of ~300 L — it wastes capacity modelling intercepts
#   - Normalising to ratio ≈ 1.0 lets the model focus purely on *relative*
#     within-vehicle behaviour (when does this vehicle burn more than usual?)
#   - Scale is then trivially restored: pred = pred_norm * veh_mean

# Step 1: per-vehicle mean from full data (locked here, reused everywhere)
veh_avg_for_norm = (
    data.groupby('vehicle')['acons']
    .mean()
    .rename('veh_avg_for_norm')
)
global_mean_for_norm = float(data['acons'].mean())

data = data.join(veh_avg_for_norm, on='vehicle')

# Step 2: normalised target
data['target_norm'] = data['acons'] / data['veh_avg_for_norm'].clip(lower=1.0)

# Step 3: clip to [0.05, 5.0] — removes statistical outliers from target space
# 0.05 = vehicle consuming 5% of its average (near-zero shift, keep some signal)
# 5.0  = vehicle consuming 5× its average (genuine extreme, not a data error)
NORM_CLIP_LO = 0.05
NORM_CLIP_HI = 5.0
data['target_norm'] = data['target_norm'].clip(NORM_CLIP_LO, NORM_CLIP_HI)

print("\n[NEW PART 11] Normalised target diagnostics:")
print(data['target_norm'].describe().round(4))
print(f"  Rows clipped at {NORM_CLIP_LO}: {(data['target_norm'] <= NORM_CLIP_LO).sum()}")
print(f"  Rows clipped at {NORM_CLIP_HI}: {(data['target_norm'] >= NORM_CLIP_HI).sum()}")
print(f"  Target mean (expect ≈ 1.0) : {data['target_norm'].mean():.4f}")
print(f"  Target std  (expect < raw) : {data['target_norm'].std():.4f}  "
      f"vs raw std {data['acons'].std():.1f}")

data[['vehicle','date','shift','acons','lag_acons_1','ewm_acons_3','acons_trend_3_7','workload']].head(8)

Building lag, rolling, EWM and advanced features...
  ewm_acons_3 created
  ewm_acons_5 created
  ewm_acons_10 created
  history_length + is_cold_start created
  workload, idle_x_dist, lift_x_dist created
  high_load_flag created  (dist_q80=73.4 km, idle_q80=2.10 h, flagged=35.7%)
  squared + log features created
  shift interaction features created
  vehicle×shift mean encoding created
  analog signal flag + dump feature nulls handled

Dataset with all features : (3819, 95)
Cold-start rows           : 96

[NEW PART 11] Normalised target diagnostics:
count    3819.0000
mean        1.0026
std         0.5587
min         0.0500
25%         0.7096
50%         1.0438
75%         1.2997
max         4.3285
Name: target_norm, dtype: float64
  Rows clipped at 0.05: 358
  Rows clipped at 5.0: 0
  Target mean (expect ≈ 1.0) : 1.0026
  Target std  (expect < raw) : 0.5587  vs raw std 97.1


,vehicle,date,shift,acons,lag_acons_1,ewm_acons_3,acons_trend_3_7,workload
0,Dump002,2026-01-01,A,6.76,NaN,NaN,NaN,0.360292
1,Dump002,2026-01-02,A,5.41,6.76,6.760000,0.000000,0.140055
2,Dump002,2026-01-04,A,126.22,5.41,5.860000,0.000000,51.350163
3,Dump002,2026-01-04,B,74.36,126.22,74.637143,0.000000,35.014069
4,Dump002,2026-01-05,A,166.65,74.36,74.489333,15.475833,63.918053
5,Dump002,2026-01-05,B,186.85,166.65,122.056129,46.530000,77.367592
6,Dump002,2026-01-06,A,64.38,186.85,154.967302,48.245000,22.244627
7,Dump002,2026-01-06,B,131.53,64.38,109.317008,49.203333,57.140228


## Block 6 — [IMPROVED] Encode Categoricals + Build Feature List

Using pandas `category` dtype (avoids LabelEncoder ordering artefact).
Mapping dicts stored explicitly for deterministic reuse on test.

In [11]:
# [IMPROVED PART 6] category dtype — CatBoost handles natively (integer codes)
data['vehicle_enc'] = data['vehicle'].astype('category').cat.codes
vehicle_cat_map     = dict(zip(data['vehicle'], data['vehicle_enc']))

data['shift_enc'] = data['shift'].map({'C': 0, 'A': 1, 'B': 2}).fillna(0).astype(int)

# Fleet categorical columns
fleet_cat_cols = []
fleet_le_maps  = {}   # str → int, stored for test reuse
for col in data.columns:
    if data[col].dtype == object and col not in ['vehicle','date','shift']:
        data[col]          = data[col].astype('category')
        # Build str→code mapping
        fleet_le_maps[col] = {cat: code for code, cat in enumerate(data[col].cat.categories)}
        data[col]          = data[col].cat.codes
        fleet_cat_cols.append(col)
        print(f"  category encoded: {col}")

# Feature list
BASE_FEATURES = [
    # Physics
    'total_dist_km', 'positive_lift_m', 'negative_lift_m', 'net_lift',
    'alt_range', 'alt_std',
    'grad_mean', 'grad_abs_mean', 'grad_std', 'grad_max',
    'terrain_difficulty',
    # Speed
    'speed_mean', 'speed_max', 'speed_std', 'speed_p75',
    # [NEW PART 5] Speed bins
    'spd_low_frac', 'spd_med_frac', 'spd_high_frac',
    # Operating time
    'moving_hours', 'idle_hours', 'idle_fraction',
    # Behaviour
    'stop_go_events', 'stop_go_per_km', 'accel_roughness',
    # [NEW PART 5] Stop stats
    'stop_dur_mean', 'stop_dur_max',
    # [NEW PART 5] Route variability
    'lat_std', 'lon_std',
    # [NEW PART 5] Jerk
    'jerk_mean', 'jerk_max', 'jerk_std',
    # Composite
    'efficiency', 'effort', 'idle_per_km', 'lift_per_km',
    # [NEW PART 3] Workload + interactions
    'workload', 'fuel_per_km_lag1', 'idle_x_dist', 'lift_x_dist',
    # Nonlinear
    'total_dist_km_sq', 'total_dist_km_log',
    'positive_lift_m_sq', 'positive_lift_m_log',
    'idle_hours_sq', 'idle_hours_log',
    'workload_sq', 'workload_log',
    # [NEW] High-load trigger (Part 2)
    'high_load_flag',
    # Lag / rolling (MOST IMPORTANT)
    'lag_acons_1', 'lag_acons_2', 'lag_acons_3',
    'rolling_acons_3', 'rolling_acons_7', 'rolling_acons_std3',
    'lag_dist_1', 'lag_idle_1', 'rolling_dist_3',
    # [NEW PART 2] EWM + trend
    'ewm_acons_3', 'ewm_acons_5', 'ewm_acons_10',
    'acons_trend_3_7', 'acons_trend_sign',
    # [NEW PART 2] History
    'history_length', 'is_cold_start',
    # Vehicle stats
    'vehicle_avg_acons', 'vehicle_std_acons', 'vehicle_avg_dist',
    'dist_ratio', 'acons_ratio',
    # [NEW PART 4] Shift interactions
    'shift_x_dist', 'shift_x_idle', 'shift_x_lift', 'shift_x_lag1', 'shift_x_weekend',
    # [NEW] Vehicle×shift encoding
    'veh_shift_avg_acons',
    # Time
    'day_of_week', 'is_weekend', 'month', 'day_of_month',
    # Identity
    'shift_enc', 'vehicle_enc', 'ping_count',
    # [NEW PART 9] Analog dump signal
    'has_analog_signal',
    'dump_count', 'dump_signal_frac',
    'dump_tilt_mean', 'dump_tilt_max', 'has_analog_data',
]

# Add any fleet columns with sufficient data
for col in fleet_cat_cols + [c for c in fleet.columns if c != 'vehicle']:
    if col in data.columns and col not in BASE_FEATURES:
        if data[col].notna().sum() > 50:
            BASE_FEATURES.append(col)
            print(f"  + fleet feature: {col}")

FEATURE_COLS = [c for c in BASE_FEATURES if c in data.columns]
print(f"\nTotal features : {len(FEATURE_COLS)}")
print(FEATURE_COLS)

  category encoded: fleet
  category encoded: mine_anon
  + fleet feature: fleet
  + fleet feature: mine_anon
  + fleet feature: tankcap
  + fleet feature: dump_switch

Total features : 92
['total_dist_km', 'positive_lift_m', 'negative_lift_m', 'net_lift', 'alt_range', 'alt_std', 'grad_mean', 'grad_abs_mean', 'grad_std', 'grad_max', 'terrain_difficulty', 'speed_mean', 'speed_max', 'speed_std', 'speed_p75', 'spd_low_frac', 'spd_med_frac', 'spd_high_frac', 'moving_hours', 'idle_hours', 'idle_fraction', 'stop_go_events', 'stop_go_per_km', 'accel_roughness', 'stop_dur_mean', 'stop_dur_max', 'lat_std', 'lon_std', 'jerk_mean', 'jerk_max', 'jerk_std', 'efficiency', 'effort', 'idle_per_km', 'lift_per_km', 'workload', 'fuel_per_km_lag1', 'idle_x_dist', 'lift_x_dist', 'total_dist_km_sq', 'total_dist_km_log', 'positive_lift_m_sq', 'positive_lift_m_log', 'idle_hours_sq', 'idle_hours_log', 'workload_sq', 'workload_log', 'high_load_flag', 'lag_acons_1', 'lag_acons_2', 'lag_acons_3', 'rolling_acons_3

## Block 7 — [NEW] Time-Series Cross-Validation

Multiple rolling cutoffs — no leakage, mirrors real deployment. Uses CatBoost.

In [12]:
# [NEW PART 7] Time-series CV with rolling cutoffs — CatBoost
CV_CUTOFFS = [
    ('2026-01-31', '2026-02-28'),   # train Jan,              val Feb
    ('2026-02-14', '2026-02-28'),   # train Jan+first-half Feb, val 2nd-half Feb
    ('2026-02-28', '2026-03-15'),   # train Jan+Feb,          val first-half Mar
]

def ts_cv_score(df, feature_cols, cutoffs, seeds=[42], log_target=True):
    """[NEW] Time-series CV with CatBoost. Returns list of per-fold RMSE."""
    cv_scores = []
    for i, (train_end, val_end) in enumerate(cutoffs):
        tr = df[df['date'] <  train_end].copy()
        va = df[(df['date'] >= train_end) & (df['date'] <= val_end)].copy()
        if len(va) < 10:
            print(f"  Fold {i+1}: too few val rows ({len(va)}) — skip")
            continue

        X_tr = tr[feature_cols]
        X_va = va[feature_cols]
        y_tr = np.log1p(tr['acons']) if log_target else tr['acons']
        y_va = np.log1p(va['acons']) if log_target else va['acons']

        fold_preds = []
        for seed in seeds:
            m = CatBoostRegressor(
                iterations=3000,
                learning_rate=0.03,
                depth=8,
                min_data_in_leaf=8,
                subsample=0.8,
                colsample_bylevel=0.8,
                l2_leaf_reg=0.5,
                random_seed=seed,
                thread_count=-1,
                verbose=0,
                early_stopping_rounds=100,
            )
            m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
            p = np.expm1(m.predict(X_va)) if log_target else m.predict(X_va)
            fold_preds.append(p)

        preds  = np.mean(fold_preds, axis=0)
        actual = np.expm1(y_va) if log_target else y_va.values
        rmse   = np.sqrt(mean_squared_error(actual, preds))
        cv_scores.append(rmse)
        print(f"  Fold {i+1} [{train_end} → {val_end}] : RMSE = {rmse:.3f}  (n_val={len(va)})")

    if cv_scores:
        print(f"\nCV RMSE mean = {np.mean(cv_scores):.3f}  std = {np.std(cv_scores):.3f}")
    return cv_scores

cv_scores = ts_cv_score(data, FEATURE_COLS, CV_CUTOFFS, seeds=[42])

  Fold 1 [2026-01-31 → 2026-02-28] : RMSE = 28.844  (n_val=1567)
  Fold 2 [2026-02-14 → 2026-02-28] : RMSE = 20.995  (n_val=571)
  Fold 3 [2026-02-28 → 2026-03-15] : RMSE = 26.228  (n_val=879)

CV RMSE mean = 25.355  std = 3.263


## Block 8 — Train/Validate Split

In [13]:
train_df = data[data['date'] <  '2026-03-01'].copy()
val_df   = data[data['date'] >= '2026-03-01'].copy()

print(f"Train : {len(train_df)} rows  ({train_df['date'].min().date()} → {train_df['date'].max().date()})")
print(f"Val   : {len(val_df)} rows  ({val_df['date'].min().date()} → {val_df['date'].max().date()})")
print(f"\nTrain acons: mean={train_df['acons'].mean():.1f}  std={train_df['acons'].std():.1f}")
print(f"Val   acons: mean={val_df['acons'].mean():.1f}  std={val_df['acons'].std():.1f}")

X_train = train_df[FEATURE_COLS]
X_val   = val_df[FEATURE_COLS]
y_train_log = np.log1p(train_df['acons'])
y_val_log   = np.log1p(val_df['acons'])
y_train_raw = train_df['acons']
y_val_raw   = val_df['acons']

# [NEW PART 11] Normalised targets — propagate veh_avg_for_norm into splits
# veh_avg_for_norm was joined onto `data` in Block 5; slicing here preserves it.
# We also expose the per-vehicle lookup dict used at inference.
y_train_norm = train_df['target_norm']
y_val_norm   = val_df['target_norm']

# Build the per-vehicle scale lookup dict (used in inference to recover scale)
# Keyed on vehicle string → float mean acons from FULL data (not train-only)
veh_norm_dict = veh_avg_for_norm.to_dict()   # vehicle → mean acons

print(f"\nX_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"\n[NEW PART 11] Normalised target stats (train):")
print(f"  mean = {y_train_norm.mean():.4f}  (expect ≈ 1.0)")
print(f"  std  = {y_train_norm.std():.4f}")
print(f"  min  = {y_train_norm.min():.4f}  max = {y_train_norm.max():.4f}")

Train : 2940 rows  (2026-01-01 → 2026-02-20)
Val   : 879 rows  (2026-03-01 → 2026-03-11)

Train acons: mean=171.4  std=95.5
Val   acons: mean=174.8  std=102.1

X_train shape : (2940, 92)
X_val shape   : (879, 92)

[NEW PART 11] Normalised target stats (train):
  mean = 0.9919  (expect ≈ 1.0)
  std  = 0.5488
  min  = 0.0500  max = 4.3285


## Block 9 — [IMPROVED] Model Training: CatBoost (Primary) + XGB + LGB

**New:** CatBoost replaces LightGBM as primary model. Multi-seed averaging, dual log+raw target blending, and normalised-target models.

In [14]:
# ── CatBoost: multi-seed + dual-target blend ─────────────────────

SEEDS = [42, 100, 2024]

CAT_PARAMS = dict(
    iterations        = 5000,
    learning_rate     = 0.02,
    depth             = 8,              # CatBoost uses depth (symmetric trees)
    min_data_in_leaf  = 5,
    subsample         = 0.9,
    colsample_bylevel = 0.9,
    l2_leaf_reg       = 0.5,
    thread_count      = -1,
    verbose           = 0,
    early_stopping_rounds = 200,
)

# =========================
# 1. LOG TARGET MODELS
# =========================
cat_preds_val_log = []
cat_models_log    = []

for seed in SEEDS:
    print(f"CAT log-target seed={seed}...", end=' ')

    model = CatBoostRegressor(**CAT_PARAMS, random_seed=seed)
    model.fit(
        X_train, np.log1p(train_df['acons']),
        eval_set=(X_val, np.log1p(val_df['acons'])),
        verbose=0,
    )

    pred_log = model.predict(X_val)
    cat_preds_val_log.append(pred_log)
    cat_models_log.append(model)

    rmse = np.sqrt(mean_squared_error(val_df['acons'], np.expm1(pred_log)))
    print(f"RMSE={rmse:.2f}")

cat_val_pred_log = np.expm1(np.mean(cat_preds_val_log, axis=0))


# =========================
# 2. RAW TARGET MODELS
# =========================
cat_preds_val_raw = []
cat_models_raw    = []

for seed in SEEDS:
    print(f"CAT raw-target seed={seed}...", end=' ')

    model = CatBoostRegressor(**CAT_PARAMS, random_seed=seed)
    model.fit(
        X_train, train_df['acons'],
        eval_set=(X_val, val_df['acons']),
        verbose=0,
    )

    pred_raw = model.predict(X_val)
    cat_preds_val_raw.append(pred_raw)
    cat_models_raw.append(model)

    rmse = np.sqrt(mean_squared_error(val_df['acons'], pred_raw))
    print(f"RMSE={rmse:.2f}")

cat_val_pred_raw = np.mean(cat_preds_val_raw, axis=0)


# =========================
# 3. FINAL 2-WAY BLEND (log + raw)
# =========================
cat_val_blend = (
    0.3 * cat_val_pred_log +
    0.7 * np.clip(cat_val_pred_raw, 0, None)
)

rmse_cat = np.sqrt(mean_squared_error(val_df['acons'], cat_val_blend))
baseline = np.sqrt(mean_squared_error(
    val_df['acons'],
    np.full(len(val_df), val_df['acons'].mean())
))

print("\n" + "="*55)
print(f"CAT (log + raw blend) RMSE : {rmse_cat:.3f}")
print(f"Naive baseline             : {baseline:.3f}")
print(f"Improvement                : {(baseline-rmse_cat)/baseline*100:.1f}%")
print("="*55)


# =========================
# 4. [NEW PART 11] NORMALISED TARGET MODELS
# =========================
cat_preds_val_norm = []
cat_models_norm    = []

for seed in SEEDS:
    print(f"CAT norm-target seed={seed}...", end=' ')

    model = CatBoostRegressor(**CAT_PARAMS, random_seed=seed)
    model.fit(
        X_train, y_train_norm,
        eval_set=(X_val, y_val_norm),
        verbose=0,
    )

    pred_norm_val = model.predict(X_val)
    cat_preds_val_norm.append(pred_norm_val)
    cat_models_norm.append(model)

    val_veh_means  = val_df['veh_avg_for_norm'].values
    pred_recovered = np.clip(pred_norm_val, 0, None) * val_veh_means
    rmse_norm      = np.sqrt(mean_squared_error(val_df['acons'], pred_recovered))
    print(f"RMSE={rmse_norm:.2f}")

cat_val_pred_norm_avg    = np.mean(cat_preds_val_norm, axis=0)
val_veh_means            = val_df['veh_avg_for_norm'].values
cat_val_pred_norm_scaled = np.clip(cat_val_pred_norm_avg, 0, None) * val_veh_means


# =========================
# 5. [NEW PART 11] FOUR-WAY BLEND
# =========================
BLEND_W_LOG  = 0.20
BLEND_W_RAW  = 0.55
BLEND_W_NORM = 0.25

cat_val_blend_4way = (
    BLEND_W_LOG  * cat_val_pred_log +
    BLEND_W_RAW  * np.clip(cat_val_pred_raw, 0, None) +
    BLEND_W_NORM * cat_val_pred_norm_scaled
)

rmse_4way = np.sqrt(mean_squared_error(val_df['acons'], cat_val_blend_4way))
print("\n" + "="*55)
print(f"CAT (log + raw blend)        RMSE : {rmse_cat:.3f}")
print(f"CAT (log + raw + norm blend) RMSE : {rmse_4way:.3f}")
delta = rmse_cat - rmse_4way
print(f"Norm model contribution           : {delta:+.3f}  "
      f"({'✓ improved' if delta > 0 else '— marginal, check weights'})")
print("="*55)

CAT log-target seed=42... RMSE=25.94
CAT log-target seed=100... RMSE=25.81
CAT log-target seed=2024... RMSE=25.99
CAT raw-target seed=42... RMSE=24.83
CAT raw-target seed=100... RMSE=24.51
CAT raw-target seed=2024... RMSE=24.83

CAT (log + raw blend) RMSE : 24.859
Naive baseline             : 102.048
Improvement                : 75.6%
CAT norm-target seed=42... RMSE=25.50
CAT norm-target seed=100... RMSE=25.69
CAT norm-target seed=2024... RMSE=25.56

CAT (log + raw blend)        RMSE : 24.859
CAT (log + raw + norm blend) RMSE : 24.734
Norm model contribution           : +0.125  (✓ improved)


## Block 10 — [ADVANCED] Sequential Inference with Blended History Update

**Part 1 fix:** history is updated with a blended value, not the raw prediction, dampening error drift.

**Part 8:** predictions that deviate more than 2.5σ from the vehicle mean are shrunk toward the mean.

In [15]:
id_mapping = pd.read_csv(DATA_PATH + "id_mapping_new.csv")
if 'date_dpr' in id_mapping.columns:  id_mapping = id_mapping.rename(columns={'date_dpr': 'date'})
if 'shift_dpr' in id_mapping.columns: id_mapping = id_mapping.rename(columns={'shift_dpr': 'shift'})
id_mapping['date'] = pd.to_datetime(id_mapping['date'])

print(f"Submission rows : {len(id_mapping)}")
print(f"Columns         : {id_mapping.columns.tolist()}")
print(f"Date range      : {id_mapping['date'].min().date()} → {id_mapping['date'].max().date()}")
print(f"Unique vehicles : {id_mapping['vehicle'].nunique()}")
id_mapping.head(6)

Submission rows : 1735
Columns         : ['id', 'vehicle', 'date', 'shift']
Date range      : 2026-01-21 → 2026-03-12
Unique vehicles : 32


,id,vehicle,date,shift
0,0,Dump002,2026-01-21,C
1,1,Dump002,2026-01-21,A
2,2,Dump002,2026-01-21,B
3,3,Dump002,2026-01-22,C
4,4,Dump002,2026-01-22,A
5,5,Dump002,2026-01-22,B


In [16]:
# Build test dataframe
test = id_mapping.copy()

# Merge telemetry
if not tele_features.empty:
    test = pd.merge(test, tele_features, on=['vehicle','date','shift'], how='left')

# Merge fleet (drop existing to avoid _x/_y)
if len(fleet) > 0:
    existing_fleet = [c for c in fleet.columns if c != 'vehicle' and c in test.columns]
    test = test.drop(columns=existing_fleet, errors='ignore')
    test = pd.merge(test, fleet, on='vehicle', how='left')

# Join vehicle stats from training
test = test.join(veh_avg,      on='vehicle')
test = test.join(veh_std,      on='vehicle')
test = test.join(veh_avg_dist, on='vehicle')

# Encode vehicle
test['vehicle_enc'] = test['vehicle'].map(vehicle_cat_map).fillna(0).astype(int)

# Fleet categoricals
for col in fleet_cat_cols:
    if col in test.columns and test[col].dtype == object:
        test[col] = test[col].astype(str).map(fleet_le_maps.get(col, {})).fillna(0).astype(int)

# Time features
test['day_of_week']  = test['date'].dt.dayofweek
test['is_weekend']   = (test['day_of_week'] >= 5).astype(int)
test['month']        = test['date'].dt.month
test['day_of_month'] = test['date'].dt.day
test['shift_num']    = test['shift'].map({'C': 0, 'A': 1, 'B': 2}).fillna(0).astype(int)
test['shift_enc']    = test['shift_num']  # 0=C, 1=A, 2=B

# Sort chronologically for sequential inference
test = test.sort_values(['vehicle','date','shift_num']).reset_index(drop=True)

# Ensure all telemetry-derived cols exist
for col in ['total_dist_km','positive_lift_m','negative_lift_m',
            'moving_hours','idle_hours','stop_go_events',
            'spd_low_frac','spd_med_frac','spd_high_frac',
            'stop_dur_mean','stop_dur_max','lat_std','lon_std',
            'jerk_mean','jerk_max','jerk_std']:
    if col not in test.columns:
        test[col] = np.nan

# Core derived features on test
test['stop_go_per_km']     = test['stop_go_events'] / (test['total_dist_km'].fillna(0) + 0.1)
test['idle_fraction']      = test['idle_hours'] / (test['moving_hours'] + test['idle_hours'] + 0.01)
test['terrain_difficulty'] = test['positive_lift_m'] / (test['total_dist_km'].fillna(0) + 0.1)
test['net_lift']           = test['positive_lift_m'].fillna(0) - test['negative_lift_m'].fillna(0)
test['efficiency']         = test['total_dist_km'] / (test['moving_hours'].fillna(0) + 0.01)
test['effort']             = test['total_dist_km'].fillna(0) * (1 + test['idle_fraction'].fillna(0) + test['terrain_difficulty'].fillna(0))
test['idle_per_km']        = test['idle_hours'] / (test['total_dist_km'].fillna(0) + 0.1)
test['lift_per_km']        = test['positive_lift_m'] / (test['total_dist_km'].fillna(0) + 0.1)

# [NEW PART 3] Workload + interactions
test['workload']    = test['total_dist_km'].fillna(0) + test['idle_hours'].fillna(0)*10 + test['positive_lift_m'].fillna(0)/100
test['idle_x_dist'] = test['idle_hours'].fillna(0) * test['total_dist_km'].fillna(0)
test['lift_x_dist'] = test['positive_lift_m'].fillna(0) * test['total_dist_km'].fillna(0)

# [NEW PART 3] Nonlinear transforms
for col in ['total_dist_km','positive_lift_m','idle_hours','workload']:
    test[f'{col}_sq']  = test[col].fillna(0) ** 2
    test[f'{col}_log'] = np.log1p(test[col].fillna(0).clip(lower=0))

# [NEW] High-load flag on test — use training quantile thresholds (no leakage)
test['high_load_flag'] = (
    (test['total_dist_km'].fillna(0) > _dist_q80) |
    (test['idle_hours'].fillna(0)    > _idle_q80)
).astype(int)
print(f"  test high_load_flag: {test['high_load_flag'].mean()*100:.1f}% flagged")

# Placeholder shift-interaction cols (computed per-row in loop)
for scol in ['shift_x_dist','shift_x_idle','shift_x_lift','shift_x_lag1','shift_x_weekend']:
    test[scol] = 0.0


# [NEW PART 9] Analog signal on test set
ANALOG_SIGNAL_START = pd.Timestamp('2026-02-17')
test['has_analog_signal'] = (test['date'] >= ANALOG_SIGNAL_START).astype(int)
for col in ['dump_count', 'dump_signal_frac', 'dump_tilt_mean', 'dump_tilt_max', 'has_analog_data']:
    if col not in test.columns:
        test[col] = np.nan
    # For rows before cutoff: no sensor, fill with 0
    test.loc[test['has_analog_signal'] == 0, col] = test.loc[test['has_analog_signal'] == 0, col].fillna(0)
print(f"Test feature shape : {test.shape}")
print(f"Telemetry match    : {test['total_dist_km'].notna().sum()} / {len(test)}")

  test high_load_flag: 31.8% flagged
Test feature shape : (1735, 77)
Telemetry match    : 1701 / 1735


In [17]:
# ── Build per-vehicle history from training ────────────────────────────
history_full = data[['vehicle','date','shift_num','acons',
                     'total_dist_km','idle_hours']].copy()
history_full = history_full.sort_values(['vehicle','date','shift_num'])

vehicle_history = defaultdict(list)
for _, row in history_full.iterrows():
    vehicle_history[row['vehicle']].append({
        'acons':         row['acons'],
        'total_dist_km': row['total_dist_km'],
        'idle_hours':    row['idle_hours'],
    })

# Precompute lookup dicts
veh_avg_dict      = veh_avg.to_dict()
veh_avg_dist_dict = veh_avg_dist.to_dict() if not veh_avg_dist.empty else {}
veh_std_dict      = veh_std.to_dict()
veh_shift_avg_dict = veh_shift_avg.set_index(['vehicle','shift'])['veh_shift_avg_acons'].to_dict()

print(f"Vehicles in history : {len(vehicle_history)}")
print(f"Total history rows  : {sum(len(v) for v in vehicle_history.values())}")

Vehicles in history : 32
Total history rows  : 3819


In [18]:
# ── [ADVANCED] Enhanced lags_from_hist: lag + rolling + EWM + trend ─────
def lags_from_hist(hist):
    """[IMPROVED] Extract all lag/rolling/EWM/trend features from ordered history."""
    vals      = [h['acons']         for h in hist]
    dist_vals = [h['total_dist_km'] for h in hist]
    idle_vals = [h['idle_hours']    for h in hist]
    n = len(vals)

    feats = {
        'lag_acons_1'       : vals[-1]               if n >= 1 else np.nan,
        'lag_acons_2'       : vals[-2]               if n >= 2 else np.nan,
        'lag_acons_3'       : vals[-3]               if n >= 3 else np.nan,
        'rolling_acons_3'   : np.mean(vals[-3:])     if n >= 1 else np.nan,
        'rolling_acons_7'   : np.mean(vals[-7:])     if n >= 1 else np.nan,
        'rolling_acons_std3': np.std(vals[-3:])      if n >= 3 else np.nan,
        'lag_dist_1'        : dist_vals[-1]           if n >= 1 else np.nan,
        'lag_idle_1'        : idle_vals[-1]           if n >= 1 else np.nan,
        'rolling_dist_3'    : np.mean(dist_vals[-3:]) if n >= 1 else np.nan,
        'history_length'    : n,
        'is_cold_start'     : int(n < 3),
    }

    # [NEW PART 2] EWM
    if n >= 1:
        s = pd.Series(vals)
        for span in [3, 5, 10]:
            feats[f'ewm_acons_{span}'] = float(s.ewm(span=span, min_periods=1).mean().iloc[-1])
    else:
        for span in [3, 5, 10]:
            feats[f'ewm_acons_{span}'] = np.nan

    # [NEW PART 2] Trend
    r3 = feats['rolling_acons_3']
    r7 = feats['rolling_acons_7']
    if not np.isnan(r3) and not np.isnan(r7):
        feats['acons_trend_3_7']  = r3 - r7
        feats['acons_trend_sign'] = float(np.sign(r3 - r7))
    else:
        feats['acons_trend_3_7']  = np.nan
        feats['acons_trend_sign'] = 0.0

    return feats


# ── [NEW PART 1] Blended history update ─────────────────────────────
def blended_history_update(pred, hist, veh_avg_acons,
                           w_pred=BLEND_PRED, w_lag=BLEND_LAG1, w_avg=BLEND_VEHAVG):
    """
    Blend pred with last history value and vehicle average to dampen drift.
    new_val = w_pred*pred + w_lag*last_acons + w_avg*vehicle_avg
    """
    lag1    = hist[-1]['acons'] if hist else veh_avg_acons
    return float(w_pred * pred + w_lag * lag1 + w_avg * veh_avg_acons)


# ── [NEW PART 8] Prediction stability ──────────────────────────────
def stabilize_pred(pred, vehicle, veh_avg_dict, veh_std_dict,
                  threshold=STABILITY_THRESHOLD, alpha=STABILITY_ALPHA):
    """
    Shrink predictions that deviate more than `threshold` std-devs
    from the vehicle historical mean toward that mean.
    """
    v_avg = veh_avg_dict.get(vehicle, pred)
    v_std = veh_std_dict.get(vehicle, 1e9)
    if v_std < 1e-6:
        return pred
    z = abs(pred - v_avg) / v_std
    if z > threshold:
        return alpha * pred + (1 - alpha) * v_avg
    return pred


print("Helper functions defined: lags_from_hist, blended_history_update, stabilize_pred")

Helper functions defined: lags_from_hist, blended_history_update, stabilize_pred


In [19]:
# ── [MODIFIED PART 11] CatBoost sequential inference with norm-target blend ──

# Ensure all features exist
for col in FEATURE_COLS:
    if col not in test.columns:
        test[col] = np.nan

all_predictions = []

for idx, row in test.iterrows():
    vehicle = row['vehicle']
    shift   = row['shift']
    hist    = vehicle_history[vehicle]

    veh_avg_acons = veh_avg_dict.get(vehicle, float(data['acons'].mean()))
    veh_avg_d     = veh_avg_dist_dict.get(
        vehicle,
        float(data['total_dist_km'].mean()) if 'total_dist_km' in data.columns else 0.0
    )

    # ── Step 1: lag features ──
    lf = lags_from_hist(hist)

    # ── Step 2: feature construction ──
    row_feats = row.to_dict()
    row_feats.update(lf)

    lag1  = lf.get('lag_acons_1', np.nan)
    lag_d = lf.get('lag_dist_1',  np.nan)

    row_feats['acons_ratio'] = (
        lag1 / (veh_avg_acons + 0.1) if pd.notna(lag1) else 1.0
    )
    row_feats['dist_ratio'] = (
        row_feats.get('total_dist_km', 0) / (veh_avg_d + 0.1)
    )
    row_feats['veh_shift_avg_acons'] = veh_shift_avg_dict.get(
        (vehicle, shift), veh_avg_acons
    )
    row_feats['fuel_per_km_lag1'] = (
        lag1 / (lag_d + 0.1)
        if (pd.notna(lag1) and pd.notna(lag_d)) else np.nan
    )

    # Shift interactions
    snum = row_feats.get('shift_num', 1)
    row_feats['shift_x_dist']    = snum * row_feats.get('total_dist_km', 0)
    row_feats['shift_x_idle']    = snum * row_feats.get('idle_hours', 0)
    row_feats['shift_x_lift']    = snum * row_feats.get('positive_lift_m', 0)
    row_feats['shift_x_lag1']    = snum * (lag1 if pd.notna(lag1) else 0)
    row_feats['shift_x_weekend'] = snum * row_feats.get('is_weekend', 0)

    row_feats['high_load_flag'] = int(
        (row_feats.get('total_dist_km', 0) > _dist_q80) or
        (row_feats.get('idle_hours',    0) > _idle_q80)
    )

    row_df = pd.DataFrame([row_feats])[FEATURE_COLS]

    # ── Step 3: CatBoost prediction — log + raw + norm blend ──
    try:
        # Log models
        cat_log_preds = [np.expm1(m.predict(row_df)[0]) for m in cat_models_log]

        # Raw models
        cat_raw_preds = [m.predict(row_df)[0] for m in cat_models_raw]

        # [NEW PART 11] Normalised models — predict ratio, then recover scale
        veh_mean_for_scale = veh_norm_dict.get(vehicle, global_mean_for_norm)
        cat_norm_preds = [
            max(0.0, m.predict(row_df)[0]) * veh_mean_for_scale
            for m in cat_models_norm
        ]

        # [MODIFIED PART 11] Four-way blend: log=0.20, raw=0.55, norm=0.25
        pred = (
            BLEND_W_LOG  * np.mean(cat_log_preds) +
            BLEND_W_RAW  * max(0.0, np.mean(cat_raw_preds)) +
            BLEND_W_NORM * np.mean(cat_norm_preds)
        )

    except Exception as e:
        print(f"Prediction error at idx={idx}: {e}")
        pred = veh_avg_acons  # safe fallback

    # ── Step 4: stabilisation ──
    if 'veh_std_dict' in globals():
        pred = stabilize_pred(pred, vehicle, veh_avg_dict, veh_std_dict)

    pred = float(max(0.1, pred))  # floor
    all_predictions.append(pred)

    # ── Step 5: blended history update ──
    blended_val = blended_history_update(pred, hist, veh_avg_acons)
    vehicle_history[vehicle].append({
        'acons':         blended_val,
        'total_dist_km': row_feats.get('total_dist_km', np.nan),
        'idle_hours':    row_feats.get('idle_hours',    np.nan),
    })

# Assign predictions
test['prediction'] = all_predictions

preds_arr = np.array(all_predictions)
print("Raw prediction stats:")
print(f"  mean = {preds_arr.mean():.1f}")
print(f"  std  = {preds_arr.std():.1f}")
print(f"  min  = {preds_arr.min():.1f}")
print(f"  max  = {preds_arr.max():.1f}")
print(f"  p90  = {np.percentile(preds_arr, 90):.1f}")
print(f"  p99  = {np.percentile(preds_arr, 99):.1f}")
print(f"  >500 = {(preds_arr > 500).sum()} rows")

Raw prediction stats:
  mean = 150.2
  std  = 106.6
  min  = 1.2
  max  = 346.9
  p90  = 282.3
  p99  = 322.4
  >500 = 0 rows


## Block 11 — Build & Save Submission

In [20]:
submission = pd.merge(
    id_mapping[['id','vehicle','date','shift']],
    test[['vehicle','date','shift','prediction']],
    on=['vehicle','date','shift'],
    how='left'
)

# Fill any missing with training-derived medians (never test-derived)
train_vehicle_med = data.groupby('vehicle')['acons'].median()
train_global_med  = data['acons'].median()

submission['prediction'] = submission['prediction'].fillna(
    submission['vehicle'].map(train_vehicle_med)
)
submission['prediction'] = submission['prediction'].fillna(train_global_med)

# Clip using training distribution percentiles
clip_lo = data['acons'].quantile(0.001)
clip_hi = data['acons'].quantile(0.999)
submission['prediction'] = submission['prediction'].clip(lower=clip_lo, upper=clip_hi)

final = submission[['id','prediction']].drop_duplicates(subset=['id'])

print(f"\n{'='*50}")
print(f"Final submission shape : {final.shape}")
print(f"IDs in id_mapping      : {len(id_mapping)}")
print(f"IDs in submission      : {len(final)}")
print(f"Missing IDs            : {len(id_mapping) - len(final)}")
print(f"Null predictions       : {final['prediction'].isnull().sum()}")
print(f"{'='*50}")
print(f"Prediction stats:")
print(f"  mean = {final['prediction'].mean():.1f}")
print(f"  std  = {final['prediction'].std():.1f}")
print(f"  min  = {final['prediction'].min():.1f}")
print(f"  max  = {final['prediction'].max():.1f}")
final.head(10)


Final submission shape : (1735, 2)
IDs in id_mapping      : 1735
IDs in submission      : 1735
Missing IDs            : 0
Null predictions       : 0
Prediction stats:
  mean = 150.2
  std  = 106.6
  min  = 1.2
  max  = 346.9


,id,prediction
0,0,5.284278
1,1,159.346002
2,2,73.909827
3,3,6.056255
4,4,82.431871
5,5,61.829265
6,6,3.181583
7,7,69.735962
8,8,68.120169
9,9,3.554232


In [21]:
final.to_csv('/kaggle/working/submission.csv', index=False)

loaded = pd.read_csv('/kaggle/working/submission.csv')
print(f"Saved: /kaggle/working/submission.csv")
print(f"Rows  : {len(loaded)}")
print(f"Cols  : {loaded.columns.tolist()}")
print(f"Nulls : {loaded.isnull().sum().to_dict()}")
print("\nFirst 15 rows:")
print(loaded.head(15).to_string(index=False))

Saved: /kaggle/working/submission.csv
Rows  : 1735
Cols  : ['id', 'prediction']
Nulls : {'id': 0, 'prediction': 0}

First 15 rows:
 id  prediction
  0    5.284278
  1  159.346002
  2   73.909827
  3    6.056255
  4   82.431871
  5   61.829265
  6    3.181583
  7   69.735962
  8   68.120169
  9    3.554232
 10  140.540092
 11   54.133266
 12    2.290289
 13   30.261960
 14   55.989735
